In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# DataCenter Event Identification Supplement Files
# ============================================================

BASE_DIR = Path("./")
RESULT_DIR = Path("./result")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Input Files (모두 ./)
# ============================================================

GLOBAL_FILE = BASE_DIR / "global_spillover_spikes_SOLVPN.csv"
SEGMENT_FILE = BASE_DIR / "segmentwise_spillover_spikes_SOLVPN.csv"
PAIRWISE_SEG_FILE = BASE_DIR / "solvpn_pairwise_spillover_with_segments.csv"

# ============================================================
# Config
# ============================================================

DATE_TOLERANCE_DAYS = 0
EVENT_WINDOW_DAYS = 3

USE_ONLY_SOLVPN_TO_X = False

MIN_ABS_SPILLOVER = None
MIN_Z = 2.0

# ============================================================
# Load
# ============================================================

global_spikes = pd.read_csv(GLOBAL_FILE)
segment_spikes = pd.read_csv(SEGMENT_FILE)
pairwise_seg = pd.read_csv(PAIRWISE_SEG_FILE)

for df in [global_spikes, segment_spikes, pairwise_seg]:
    df["Date"] = pd.to_datetime(df["Date"])

# ============================================================
# 1. Spike Classification
# ============================================================

global_key = global_spikes.copy()
segment_key = segment_spikes.copy()

key_cols = ["Date", "Pair", "DirectionMetric"]

global_key["_global"] = 1
segment_key["_segment"] = 1

merged = pd.merge(
    global_key,
    segment_key[key_cols + ["_segment"]],
    on=key_cols,
    how="outer"
)

merged["_global"] = merged["_global"].fillna(0)
merged["_segment"] = merged["_segment"].fillna(0)

def classify_spike(row):
    if row["_global"] == 1 and row["_segment"] == 1:
        return "common"
    elif row["_global"] == 1 and row["_segment"] == 0:
        return "global-only"
    elif row["_global"] == 0 and row["_segment"] == 1:
        return "segment-only"
    return "unknown"

merged["SpikeClass"] = merged.apply(classify_spike, axis=1)

# ------------------------------------------------------------
# Segment detail merge
# ------------------------------------------------------------

seg_detail_cols = [
    "Date",
    "Pair",
    "DirectionMetric",
    "Source",
    "Target",
    "SegmentID",
    "SegmentStart",
    "SegmentEnd",
    "SegmentLength",
    "Spillover",
    "SegmentMean",
    "SegmentStd",
    "SegmentThreshold",
    "SegmentZ"
]

available_seg_cols = [
    c for c in seg_detail_cols
    if c in segment_spikes.columns
]

classification = pd.merge(
    merged[key_cols + ["SpikeClass"]],
    segment_spikes[available_seg_cols],
    on=key_cols,
    how="left"
)

# ------------------------------------------------------------
# Global detail merge
# ------------------------------------------------------------

global_detail_cols = [
    "Date",
    "Pair",
    "DirectionMetric",
    "Spillover",
    "GlobalMean",
    "GlobalStd",
    "GlobalThreshold",
    "GlobalZ"
]

available_global_cols = [
    c for c in global_detail_cols
    if c in global_spikes.columns
]

classification = pd.merge(
    classification,
    global_spikes[available_global_cols],
    on=key_cols,
    how="left",
    suffixes=("_Segment", "_Global")
)

classification = classification.sort_values(
    ["Date", "Pair", "DirectionMetric"]
).reset_index(drop=True)

classification_out = RESULT_DIR / "spike_classification_SOLVPN.csv"

classification.to_csv(
    classification_out,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 2. Segment Spike Summary
# ============================================================

seg_summary = segment_spikes.copy()

seg_summary["AbsSpillover"] = (
    seg_summary["Spillover"].abs()
)

segment_spike_summary = (
    seg_summary
    .groupby([
        "SegmentID",
        "SegmentStart",
        "SegmentEnd",
        "SegmentLength",
        "Pair",
        "DirectionMetric"
    ])
    .agg(
        spike_count=("Date", "count"),
        mean_spillover=("Spillover", "mean"),
        mean_abs_spillover=("AbsSpillover", "mean"),
        max_abs_spillover=("AbsSpillover", "max"),
        mean_segment_z=("SegmentZ", "mean"),
        max_segment_z=("SegmentZ", "max")
    )
    .reset_index()
)

segment_spike_summary["spike_density"] = (
    segment_spike_summary["spike_count"]
    / segment_spike_summary["SegmentLength"]
)

segment_summary_out = (
    RESULT_DIR / "segment_spike_summary_SOLVPN.csv"
)

segment_spike_summary.to_csv(
    segment_summary_out,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 3. Pairwise Direction Summary
# ============================================================

direction_summary = segment_spikes.copy()

def direction_label(metric):
    if metric == "SOLVPN_to_X":
        return "SOLVPN_to_X"
    elif metric == "X_to_SOLVPN":
        return "X_to_SOLVPN"
    elif metric == "PairwiseNet_SOLVPN_to_X":
        return "NET_SOLVPN_to_X"
    return "unknown"

direction_summary["DirectionLabel"] = (
    direction_summary["DirectionMetric"]
    .apply(direction_label)
)

pairwise_direction_summary = (
    direction_summary
    .groupby([
        "Pair",
        "Target",
        "DirectionLabel"
    ])
    .agg(
        spike_count=("Date", "count"),
        mean_spillover=("Spillover", "mean"),
        mean_segment_z=("SegmentZ", "mean"),
        max_segment_z=("SegmentZ", "max"),
        first_spike_date=("Date", "min"),
        last_spike_date=("Date", "max")
    )
    .reset_index()
    .sort_values(
        ["Pair", "spike_count"],
        ascending=[True, False]
    )
)

direction_summary_out = (
    RESULT_DIR / "pairwise_direction_summary_SOLVPN.csv"
)

pairwise_direction_summary.to_csv(
    direction_summary_out,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 4. DC Event Candidates
# ============================================================

event_candidates = classification.copy()

event_candidates = event_candidates[
    event_candidates["SpikeClass"].isin([
        "common",
        "segment-only"
    ])
].copy()

# ------------------------------------------------------------
# Direction filter
# ------------------------------------------------------------

if USE_ONLY_SOLVPN_TO_X:
    event_candidates = event_candidates[
        event_candidates["DirectionMetric"]
        == "SOLVPN_to_X"
    ].copy()

# ------------------------------------------------------------
# Z-score filter
# ------------------------------------------------------------

if "SegmentZ" in event_candidates.columns:
    event_candidates = event_candidates[
        event_candidates["SegmentZ"]
        .fillna(-np.inf) >= MIN_Z
    ].copy()

# ------------------------------------------------------------
# Spillover magnitude filter
# ------------------------------------------------------------

spill_col = (
    "Spillover_Segment"
    if "Spillover_Segment" in event_candidates.columns
    else "Spillover"
)

if (
    MIN_ABS_SPILLOVER is not None
    and spill_col in event_candidates.columns
):
    event_candidates = event_candidates[
        event_candidates[spill_col]
        .abs() >= MIN_ABS_SPILLOVER
    ].copy()

event_candidates["EventID"] = np.arange(
    1,
    len(event_candidates) + 1
)

event_candidates["EventInterpretation"] = np.select(
    [
        event_candidates["SpikeClass"]
        .eq("common"),

        event_candidates["SpikeClass"]
        .eq("segment-only")
    ],
    [
        "robust_spillover_event",
        "regime_specific_spillover_event"
    ],
    default="unclassified"
)

event_candidates_out = (
    RESULT_DIR / "dc_event_candidates_SOLVPN.csv"
)

event_candidates.to_csv(
    event_candidates_out,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 5. DC Event Mapping
# ============================================================

event_mapping = event_candidates.copy()

event_mapping["WindowStart"] = (
    event_mapping["Date"]
    - pd.to_timedelta(EVENT_WINDOW_DAYS, unit="D")
)

event_mapping["WindowEnd"] = (
    event_mapping["Date"]
    + pd.to_timedelta(EVENT_WINDOW_DAYS, unit="D")
)

event_mapping["ExternalEventLabel"] = ""
event_mapping["ExternalEventSource"] = ""
event_mapping["EconomicNarrative"] = ""

keep_cols = [
    "EventID",
    "Date",
    "WindowStart",
    "WindowEnd",
    "Pair",
    "Target",
    "DirectionMetric",
    "SpikeClass",
    "SegmentID",
    "SegmentStart",
    "SegmentEnd",
    "Spillover_Segment",
    "SegmentZ",
    "Spillover_Global",
    "GlobalZ",
    "EventInterpretation",
    "ExternalEventLabel",
    "ExternalEventSource",
    "EconomicNarrative"
]

keep_cols = [
    c for c in keep_cols
    if c in event_mapping.columns
]

event_mapping = (
    event_mapping[keep_cols]
    .sort_values([
        "Date",
        "Pair",
        "DirectionMetric"
    ])
    .reset_index(drop=True)
)

event_mapping_out = (
    RESULT_DIR / "dc_event_mapping_SOLVPN.csv"
)

event_mapping.to_csv(
    event_mapping_out,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# Summary Print
# ============================================================

print("\nSaved files:")
print(classification_out)
print(segment_summary_out)
print(direction_summary_out)
print(event_candidates_out)
print(event_mapping_out)

print("\nSpike Class Counts")
print(
    classification["SpikeClass"]
    .value_counts()
)

print("\nTop Segment Spike Summary")
print(
    segment_spike_summary
    .sort_values(
        "spike_count",
        ascending=False
    )
    .head(10)
)

print("\nTop Direction Summary")
print(
    pairwise_direction_summary
    .sort_values(
        "spike_count",
        ascending=False
    )
    .head(10)
)

print("\nEvent Candidates")
print(
    event_candidates.head(20)
)


Saved files:
result\spike_classification_SOLVPN.csv
result\segment_spike_summary_SOLVPN.csv
result\pairwise_direction_summary_SOLVPN.csv
result\dc_event_candidates_SOLVPN.csv
result\dc_event_mapping_SOLVPN.csv

Spike Class Counts
SpikeClass
segment-only    92
common          86
global-only     45
Name: count, dtype: int64

Top Segment Spike Summary
    SegmentID SegmentStart  SegmentEnd  SegmentLength  \
5           0   2022-12-01  2023-10-30          180.0   
19          0   2022-12-01  2023-10-30          180.0   
49          2   2024-07-17  2025-04-15          145.0   
47          2   2024-07-17  2025-04-15          145.0   
75          3   2025-04-16  2026-01-06          141.0   
78          3   2025-04-16  2026-01-06          141.0   
25          1   2023-10-31  2024-07-16          140.0   
10          0   2022-12-01  2023-10-30          180.0   
8           0   2022-12-01  2023-10-30          180.0   
53          2   2024-07-17  2025-04-15          145.0   

                    